# Semana 02: Controle de Versão Avançado & Intensivo GitFlow (Parte 1)

## Módulo de Gerenciamento de Código e Colaboração

Este notebook é a **Parte 1 do Intensivo sobre o modelo GitFlow**, apresentando a arquitetura de dados do Git (Grafo Acíclico Dirigido - DAG), as convenções de padronização de commits (**Conventional Commits**), o ciclo de vida completo das branches no **GitFlow** e as estratégias de mesclagem (`merge` vs `rebase`).

### Objetivos de aprendizagem
- Compreender a arquitetura interna de dados do Git (Blobs, Trees, Commits, Refs e o grafo DAG).
- Dominar a padronização de mensagens de commit com a especificação **Conventional Commits** (`feat:`, `fix:`, `refactor:`, `chore:`, `feat!:`).
- Entender o ciclo de vida e o isolamento de cada uma das 5 branches do **GitFlow** (`main`, `develop`, `feature/*`, `release/*`, `hotfix/*`).
- Diferenciar as mecânicas de mesclagem: `git merge --no-ff` (Fast-Forward desativado) vs `git rebase`.
- Executar um simulador em Python do ciclo de trabalho no GitFlow com detecção e resolução manual de conflitos.

---


## 1. Fundamentação Teórica

### 1.1 A Estrutura Interna de Dados do Git (O Grafo DAG)

Diferente de sistemas de controle de versão legados (como o Subversion/SVN), o Git não armazena diferenças em formato de *diffs* de texto delta, mas sim um histórico de **snapshots imutáveis** organizados em um Grafo Acíclico Dirigido (**DAG**):

```text
  [HEAD] -> refs/heads/feature/leitor-plc
                  |
                  v
           (Commit C3: feat: adiciona leitor S7-1500)
             | (Tree: a1b2c3d...)
             |   +-- blob: plc_driver.py
             |   +-- blob: config.json
                  |
                  v (Aponta para o Commit Pai)
           (Commit C2: fix: ajusta timeout de conexao)
                  |
                  v
           (Commit C1: initial commit)
```

#### Os 4 Objetos Principais do Git:
1. **Blob (Binary Large Object):** Armazena apenas o conteúdo do arquivo (sem metadados de permissão ou nome).
2. **Tree:** Funciona como um diretório, mapeando nomes de arquivos e permissões para seus respectivos Blobs ou sub-Trees.
3. **Commit:** Contém um ponteiro para a Tree raiz do projeto, autor, data, mensagem descritiva e os ponteiros para os commits pais (*parent commits*).
4. **Ref (Reference):** Um ponteiro simples e mutável para o hash SHA de um commit (ex: `refs/heads/main`, `refs/tags/v1.0.0`).

---

### 1.2 O Modelo GitFlow em Detalhes

Proposto por Vincent Driessen, o **GitFlow** é um modelo estrito de ramificação projetado para garantir estabilidade contínua no ambiente de produção:

![Estratégia de Ramificação GitFlow](img/gitflow_diagrama_completo.jpg)

#### O Papel das 5 Branches no GitFlow:

| Branch | Origem | Destino do Merge | Propósito & Regras |
| :--- | :--- | :--- | :--- |
| `main` | N/A | N/A | Armazena o código **oficial em produção**. Nunca recebe commits diretos; cada commit deve corresponder a uma tag de versão (`v1.0.0`). |
| `develop` | `main` | `main` (via release) | Branch principal de **integração contínua**. Reúne todas as funcionalidades concluídas para o próximo lançamento. |
| `feature/*` | `develop` | `develop` | Isolamento para desenvolvimento de **novas funcionalidades** (`feature/telemetria-oee`). Integram via Pull Request. |
| `release/*` | `develop` | `main` e `develop` | Preparação para **lançamento de nova versão** (`release/v1.1.0`). Destinada a testes de homologação, ajustes de docs e pequenos bug fixes. |
| `hotfix/*` | `main` | `main` e `develop` | Correção urgente de **bugs críticos em produção** (`hotfix/corrigir-estouro-memoria`). Ao finalizar, incrementa o número PATCH da tag. |

---

### 1.3 Padronização com Conventional Commits

A especificação **Conventional Commits** adiciona regras semânticas legíveis tanto por humanos quanto por ferramentas automatizadas de changelog:

$$\text{tipo}(\text{escopo opcional}): \text{descrição clara no imperativo}$$

#### Tipos Principais:
- `feat:` Adiciona nova funcionalidade (ex: `feat(plc): conecta leitor Siemens S7`).
- `fix:` Corrige um bug (ex: `fix(api): trata exceção de timeout na leitura`).
- `docs:` Alterações exclusivas na documentação.
- `refactor:` Alteração de código que não altera funcionalidade nem corrige bug.
- `chore:` Atualização de dependências ou tarefas de build.
- `feat!:` Indica uma **Breaking Change** (alteração incompatível que exige grande mudança na versão MAJOR).

---

### 1.4 Estratégias de Mesclagem: `git merge --no-ff` vs `git rebase`

No GitFlow, é essencial preservar o histórico visual da criação e encerramento de branches de funcionalidades:

```text
  MERGE COM FAST-FORWARD (PADRÃO):                 MERGE COM --NO-FF (GITFLOW PREFERIDO):
  develop --- C1 --- C2 --- C3 (Perde historico)   develop --- C1 -------------- M1 (Preserva o nó de merge)
                             /                                   \             /
                            /                                     +--- C2 --- C3 (feature/oee)
```

- `git merge --no-ff`: Garante que um **commit de merge** seja sempre criado, documentando visualmente quando uma funcionalidade foi integrada à branch `develop`.
- `git rebase`: Reescreve o histórico, aplicando os commits de uma branch por cima de outra. Útil para limpar commits locais antes de abrir o Pull Request.

---


## 2. Prática — Gerenciador e Validador de Fluxo GitFlow em Python

Nesta prática, executaremos uma simulação completa em Python que valida o nome das branches, valida a sintaxe dos commits e executa o algoritmo de simulação de mesclagem de arquivos com detecção e resolução de conflitos de código.

In [1]:
import re

# Simulação de validação de mensagens de commit no Git
mensagens_commits = [
    "feat(sensor): adiciona leitor de vibração em tempo real",
    "fix(banco): ajusta pool de conexões do PostgreSQL",
    "arrumei o codigo aqui", # Inválido (fora do padrão)
    "chore(deps): atualiza pacote pytest para v7.4.0",
    "feat!: altera contrato da API REST de telemetria"
]

def validar_commit_conventional(msg):
    padrao = r"^(feat|fix|docs|style|refactor|perf|test|chore)(\([a-z0-9-]+\))?!?: .+"
    return bool(re.match(padrao, msg))

print("=== VALIDAÇÃO DE CONVENTIONAL COMMITS ===\n")
for msg in mensagens_commits:
    valido = validar_commit_conventional(msg)
    status = "VÁLIDO" if valido else "INVÁLIDO"
    print(f"{status} | '{msg}'")

# Simulação de resolução de conflito de merge entre 'develop' e 'feature/oee'
conteudo_develop = ["PORTA_API = 8000", "TIMEOUT_MS = 5000", "HABILITAR_CACHE = True"]
conteudo_feature = ["PORTA_API = 9090", "TIMEOUT_MS = 5000", "HABILITAR_CACHE = False"]

def resolver_conflito_merge(base_dev, branch_feat):
    resultado = []
    for i in range(len(base_dev)):
        if base_dev[i] == branch_feat[i]:
            resultado.append(base_dev[i])
        else:
            resultado.append(f"# CONFLITO RESOLVIDO: Mantido valor da feature ({branch_feat[i]})")
    return resultado

print("\n=== ARQUIVO DE CONFIGURAÇÃO FINAL PÓS-MERGE RESOLVIDO ===")
for l in resolver_conflito_merge(conteudo_develop, conteudo_feature):
    print(l)


=== VALIDAÇÃO DE CONVENTIONAL COMMITS ===

VÁLIDO | 'feat(sensor): adiciona leitor de vibração em tempo real'
VÁLIDO | 'fix(banco): ajusta pool de conexões do PostgreSQL'
INVÁLIDO | 'arrumei o codigo aqui'
VÁLIDO | 'chore(deps): atualiza pacote pytest para v7.4.0'
VÁLIDO | 'feat!: altera contrato da API REST de telemetria'

=== ARQUIVO DE CONFIGURAÇÃO FINAL PÓS-MERGE RESOLVIDO ===
# CONFLITO RESOLVIDO: Mantido valor da feature (PORTA_API = 9090)
TIMEOUT_MS = 5000
# CONFLITO RESOLVIDO: Mantido valor da feature (HABILITAR_CACHE = False)


---

## 3. Exercícios de Fixação e Avaliação

### Questão 1
Explique a função do Grafo Acíclico Dirigido (DAG) no Git. Qual a diferença entre um objeto **Blob**, um objeto **Tree** e um objeto **Commit**?

### Questão 2
No modelo GitFlow, descreva a sequência exata de branches envolvidas na criação de uma nova funcionalidade (desde o nascimento a partir da `develop` até a mesclagem final via Pull Request).

### Questão 3
Se um problema crítico paralisa o sistema em produção, qual branch deve ser criada, a partir de qual ponto ela surge e para quais branches a correção deve ser propagada ao término?

### Questão 4
Diferencie o comportamento de `git merge --no-ff` de um `git rebase`. Por que a flag `--no-ff` é a recomendação padrão do GitFlow para mesclagens na branch `develop`?
